We're about to create and use our own MCP Server and MCP Client!

It's pretty simple, but it's not super-simple. The excitment around MCP is about how easy it is to share and use other MCP Servers - making our own does involve a bit of work.

Let's review some python code made mostly by a hard-working Engineering Team:

accounts.py

In [17]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override=True)

True

In [18]:
from accounts import Account

In [26]:
account = Account.get("Ganesh")
account

Account(name='ganesh', balance=9741.484, strategy='', holdings={'AMZN': 6}, transactions=[3 shares of AMZN at 65.13 each., 3 shares of AMZN at 21.042 each.], portfolio_value_time_series=[('2025-08-02 11:49:13', 10038.61), ('2025-08-02 11:49:16', 10056.61), ('2025-08-02 21:20:13', 10071.484), ('2025-08-02 21:20:16', 9981.484)])

In [27]:
account.buy_shares("AMZN", 3, "Because this bookstore website looks promising")

'Completed. Latest details:\n{"name": "ganesh", "balance": 9516.034, "strategy": "", "holdings": {"AMZN": 9}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 65.13, "timestamp": "2025-08-02 11:49:13", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 21.042, "timestamp": "2025-08-02 21:20:13", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 75.15, "timestamp": "2025-08-02 21:22:51", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2025-08-02 11:49:13", 10038.61], ["2025-08-02 11:49:16", 10056.61], ["2025-08-02 21:20:13", 10071.484], ["2025-08-02 21:20:16", 9981.484], ["2025-08-02 21:22:51", 10398.034]], "total_portfolio_value": 10398.034, "total_profit_loss": 398.03399999999965}'

In [29]:
account.report()

'{"name": "ganesh", "balance": 9516.034, "strategy": "", "holdings": {"AMZN": 9}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 65.13, "timestamp": "2025-08-02 11:49:13", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 21.042, "timestamp": "2025-08-02 21:20:13", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 75.15, "timestamp": "2025-08-02 21:22:51", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2025-08-02 11:49:13", 10038.61], ["2025-08-02 11:49:16", 10056.61], ["2025-08-02 21:20:13", 10071.484], ["2025-08-02 21:20:16", 9981.484], ["2025-08-02 21:22:51", 10398.034], ["2025-08-02 21:22:57", 9642.034], ["2025-08-02 21:23:02", 10164.034]], "total_portfolio_value": 10164.034, "total_profit_loss": 164.03399999999965}'

In [30]:
account.list_transactions()

[{'symbol': 'AMZN',
  'quantity': 3,
  'price': 65.13,
  'timestamp': '2025-08-02 11:49:13',
  'rationale': 'Because this bookstore website looks promising'},
 {'symbol': 'AMZN',
  'quantity': 3,
  'price': 21.042,
  'timestamp': '2025-08-02 21:20:13',
  'rationale': 'Because this bookstore website looks promising'},
 {'symbol': 'AMZN',
  'quantity': 3,
  'price': 75.15,
  'timestamp': '2025-08-02 21:22:51',
  'rationale': 'Because this bookstore website looks promising'}]

### Now we write an MCP server and use it directly!

In [37]:
# Now let's use our accounts server as an MCP server

params = {"command": "uv", "args": ["run", "accounts_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()


In [38]:
mcp_tools

[Tool(name='get_balance', title=None, description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'get_balanceOutput', 'type': 'object'}, annotations=None, meta=None),
 Tool(name='get_holdings', title=None, description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, outputSchema={'additionalProperties': {'type': 'integer'}, 'title': 'get_holdingsDictOutput', 'type': 'object'}, annotations=None, meta=None),
 Tool(name='buy_shares', title=None, description=

In [ ]:
instructions = "You are able to manage an account for a client, and answer questions about the account."
request = "My name is Ganesh and my account is under the name Ganesh. What's my balance and my holdings?"
model = "gpt-4.1-mini"

In [46]:

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="account_manager", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("account_manager"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))


Ganesh, your current account balance is $9,516.03. Is there anything specific you would like to do with your account or any other information you need?

### Now let's build our own MCP Client

In [41]:
from accounts_client import get_accounts_tools_openai, read_accounts_resource, list_accounts_tools

mcp_tools = await list_accounts_tools()
print(mcp_tools)
openai_tools = await get_accounts_tools_openai()
print(openai_tools)

[Tool(name='get_balance', title=None, description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'get_balanceOutput', 'type': 'object'}, annotations=None, meta=None), Tool(name='get_holdings', title=None, description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, outputSchema={'additionalProperties': {'type': 'integer'}, 'title': 'get_holdingsDictOutput', 'type': 'object'}, annotations=None, meta=None), Tool(name='buy_shares', title=None, description="B

In [43]:
request = "My name is Ganesh and my account is under the name Ganesh. What's my balance?"

with trace("account_mcp_client"):
    agent = Agent(name="account_manager", instructions=instructions, model=model, tools=openai_tools)
    result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Ganesh, your account balance is $9,516.03. Is there anything else you would like to do with your account?

In [44]:
context = await read_accounts_resource("Ganesh")
print(context)

{"name": "ganesh", "balance": 9516.034, "strategy": "", "holdings": {"AMZN": 9}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 65.13, "timestamp": "2025-08-02 11:49:13", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 21.042, "timestamp": "2025-08-02 21:20:13", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 75.15, "timestamp": "2025-08-02 21:22:51", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2025-08-02 11:49:13", 10038.61], ["2025-08-02 11:49:16", 10056.61], ["2025-08-02 21:20:13", 10071.484], ["2025-08-02 21:20:16", 9981.484], ["2025-08-02 21:22:51", 10398.034], ["2025-08-02 21:22:57", 9642.034], ["2025-08-02 21:23:02", 10164.034], ["2025-08-03 06:50:03", 10164.034]], "total_portfolio_value": 10164.034, "total_profit_loss": 164.03399999999965}


In [45]:
from accounts import Account
Account.get("Ganesh").report()

'{"name": "ganesh", "balance": 9516.034, "strategy": "", "holdings": {"AMZN": 9}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 65.13, "timestamp": "2025-08-02 11:49:13", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 21.042, "timestamp": "2025-08-02 21:20:13", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 75.15, "timestamp": "2025-08-02 21:22:51", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2025-08-02 11:49:13", 10038.61], ["2025-08-02 11:49:16", 10056.61], ["2025-08-02 21:20:13", 10071.484], ["2025-08-02 21:20:16", 9981.484], ["2025-08-02 21:22:51", 10398.034], ["2025-08-02 21:22:57", 9642.034], ["2025-08-02 21:23:02", 10164.034], ["2025-08-03 06:50:03", 10164.034], ["2025-08-03 06:50:53", 9813.034]], "total_portfolio_value": 9813.034, "total_profit_loss": -186.96600000000035}'